# 06 -- Straty hydrauliczne

Ten notebook przedstawia modele strat hydraulicznych w drodze wodnej MEW:

1. **Krata wlotowa** -- wzor Kirschmera
2. **Straty tarcia w rurociagu** -- rownanie Darcy'ego-Weisbacha
3. **Straty miejscowe** -- kolana, zwezenia, zasuwy
4. **Spirala turbiny** -- straty w komorze spiralnej
5. **Rura ssawna** -- straty w dyfuzorze wylotowym
6. **Skladanie modeli** -- wybor komponentow dla konkretnej instalacji

Kazdy model jest zaimplementowany w module `src/losses.py` i moze byc wybrany
niezaleznie w zaleznosci od typu instalacji (rurociag, kanal otwarty, itp.).

---

## Modul `src/losses.py`

**Prompt do LLM tworzacy ten modul:**
> *"Stworz modul src/losses.py do obliczania strat hydraulicznych w drodze wodnej MEW.
> Kazda funkcja przyjmuje przeplyw Q [m3/s] i zwraca strate spadku DeltaH [m].
> Funkcje: trash_rack_loss (wzor Kirschmera), pipe_friction_loss (Darcy-Weisbach
> z przyblizeniem Swamee-Jain), minor_loss (straty miejscowe), spiral_casing_loss,
> draft_tube_loss, total_head_loss (sumuje wybrane komponenty), net_head."*

**Funkcje w module:**
- `trash_rack_loss(Q, A_rack, ...)` -- strata na kracie wlotowej
- `pipe_friction_loss(Q, D, L, k_s)` -- tarcie w rurociagu
- `minor_loss(Q, A, xi)` -- straty miejscowe (uniwersalna)
- `spiral_casing_loss(Q, A_inlet, xi)` -- spirala turbiny
- `draft_tube_loss(Q, A_inlet, A_outlet, xi)` -- rura ssawna
- `total_head_loss(Q, loss_components)` -- suma wybranych strat
- `net_head(H_gross, Q, loss_components)` -- spad netto

## Konfiguracja

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: zaladuj modul src/losses (trash_rack_loss,
> pipe_friction_loss, minor_loss, spiral_casing_loss, draft_tube_loss,
> total_head_loss, net_head)."*

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.losses import (
    trash_rack_loss,
    pipe_friction_loss,
    minor_loss,
    spiral_casing_loss,
    draft_tube_loss,
    total_head_loss,
    net_head,
)

# Zakres przeplywow do demonstracji
Q_demo = np.linspace(1, 200, 200)

print('Modul zaladowany.')

---
## Ogolne rownanie strat hydraulicznych

Straty hydrauliczne w drodze wodnej zmniejszaja uzytkowy spad MEW.
Rozrozniamy:

- **Straty liniowe** (tarciowe) -- tarcie wody o sciany rurociagu
- **Straty miejscowe** (lokalne) -- zmiany kierunku, przekroju, przeplyw przez elementy

Obie kategorie wyraza sie za pomoca **wysokosci predkosci**:

$$\Delta H = \xi \cdot \frac{v^2}{2g}$$

gdzie:
- $\xi$ -- wspolczynnik strat [-] (rozny dla kazdego elementu)
- $v = Q / A$ -- predkosc przeplywu [m/s]
- $g = 9.81$ m/s$^2$

**Spad netto** to spad brutto pomniejszony o sume strat:

$$H_{netto} = H_{brutto} - \sum_i \Delta H_i$$

---
## Krok 1: Krata wlotowa (trash rack)

Krata wlotowa chroni turbine przed zanieczyszczeniami.
Straty obliczamy wzorem **Kirschmera**:

$$\Delta H_{krata} = \beta \cdot \left(\frac{s}{b}\right)^{4/3} \cdot \sin\alpha \cdot \frac{v^2}{2g}$$

gdzie:
- $\beta$ -- wspolczynnik ksztaltu pretow:
  - 2.42 -- prety prostokatne
  - 1.79 -- prety kolowe
  - 1.67 -- prety opływowe
- $s$ -- szerokosc preta [m]
- $b$ -- przeswit miedzy pretami [m]
- $\alpha$ -- kat nachylenia kraty [deg] (90° = pionowa)
- $v$ -- predkosc naplywu na krate [m/s]

**Prompt do LLM:**
> *"Napisz kod ktory pokaze straty na kracie wlotowej dla roznych ksztaltow pretow
> (prostokatne, kolowe, oplyniowe) w funkcji przeplywu Q. Uzyj A_rack=4m2,
> s=12mm, b=50mm. Narysuj wykres plotly."*

**Uzyte funkcje:** `src.losses.trash_rack_loss()`

In [ ]:
# Parametry kraty
A_rack = 4.0     # m2
s = 0.012         # m (12 mm)
b = 0.050         # m (50 mm)

# Porownanie ksztaltow pretow
shapes = [
    ('Prostokatne (beta=2.42)', 2.42, 'firebrick'),
    ('Kolowe (beta=1.79)', 1.79, 'royalblue'),
    ('Oplyniowe (beta=1.67)', 1.67, 'green'),
]

fig = go.Figure()
for label, beta, color in shapes:
    dH = trash_rack_loss(Q_demo, A_rack, bar_width=s, bar_spacing=b, bar_shape_coeff=beta)
    fig.add_trace(go.Scatter(x=Q_demo, y=dH * 100, mode='lines',
        name=label, line=dict(color=color, width=2)))

fig.update_layout(
    title=f'Straty na kracie wlotowej (s={s*1000:.0f}mm, b={b*1000:.0f}mm, A={A_rack}m2)',
    xaxis_title='Q [m3/s]', yaxis_title='DeltaH [cm]',
    height=400, hovermode='x unified',
)
fig.show()

# Wartosc przy typowym przeplywie
Q_typ = 50.0
dH_typ = trash_rack_loss(Q_typ, A_rack, bar_width=s, bar_spacing=b)
print(f'Strata na kracie przy Q={Q_typ} m3/s: {dH_typ:.4f} m = {dH_typ*100:.2f} cm')

---
## Krok 2: Straty tarcia w rurociagu (Darcy-Weisbach)

Dla przeplywu w rurociagu straty tarcia obliczamy rownaniem **Darcy'ego-Weisbacha**:

$$\Delta H_f = f \cdot \frac{L}{D} \cdot \frac{v^2}{2g}$$

gdzie wspolczynnik tarcia $f$ zalezy od liczby Reynoldsa i chropowatosc sciany.
Uzywamy przyblizenia **Swamee-Jain** (jawne, nie wymaga iteracji):

$$f = \frac{0.25}{\left[\log_{10}\left(\frac{k_s}{3.7D} + \frac{5.74}{Re^{0.9}}\right)\right]^2}$$

Typowe chropowatosc $k_s$:
- Rura stalowa nowa: 0.05 mm
- Rura stalowa uzywana: 0.1--1.0 mm
- Beton gladki: 0.3--1.0 mm
- Beton szorstki: 1.0--3.0 mm

**Prompt do LLM:**
> *"Napisz kod ktory pokaze straty tarcia w rurociagu dla roznych srednic
> (D=1.0, 1.5, 2.0 m) i stalej dlugosci L=100m. Narysuj wykres."*

**Uzyte funkcje:** `src.losses.pipe_friction_loss()`

In [ ]:
L = 100.0   # m
k_s = 0.001  # m (1mm, stal uzywana)

fig = go.Figure()
for D, color in [(1.0, 'firebrick'), (1.5, 'royalblue'), (2.0, 'green')]:
    dH = pipe_friction_loss(Q_demo, D=D, L=L, k_s=k_s)
    fig.add_trace(go.Scatter(x=Q_demo, y=dH, mode='lines',
        name=f'D={D} m', line=dict(color=color, width=2)))

fig.update_layout(
    title=f'Straty tarcia w rurociagu (L={L}m, ks={k_s*1000:.1f}mm)',
    xaxis_title='Q [m3/s]', yaxis_title='DeltaH [m]',
    height=400, hovermode='x unified',
)
fig.show()

# Wartosc przy typowym przeplywie
Q_typ = 50.0
for D in [1.0, 1.5, 2.0]:
    dH = pipe_friction_loss(Q_typ, D=D, L=L, k_s=k_s)
    v = Q_typ / (np.pi * D**2 / 4)
    print(f'D={D}m: v={v:.2f} m/s, DeltaH={dH:.3f} m')

---
## Krok 3: Straty miejscowe

Straty miejscowe powstaja przy kazdej zmianie kierunku lub przekroju:

$$\Delta H_{lok} = \xi \cdot \frac{v^2}{2g}$$

Typowe wspolczynniki $\xi$:

| Element | $\xi$ |
|---------|-------|
| Wlot ostrokrawedzisty | 0.5 |
| Wlot zaokraglony | 0.1--0.2 |
| Kolano 90° (r/D=1) | 0.5--1.0 |
| Kolano 90° (r/D=3) | 0.2--0.3 |
| Zasuwa (otwarta) | 0.1--0.2 |
| Zasuwa motylkowa | 0.2--0.5 |
| Nagle zwezenie | 0.5·(1-A2/A1) |
| Nagle rozszerzenie | (1-A1/A2)² |
| Wylot z rury | 1.0 |

**Prompt do LLM:**
> *"Napisz kod ktory pokaze straty miejscowe dla typowych elementow
> (wlot zaokraglony, kolano, zasuwa) przy przeplywach 1-200 m3/s
> i przekroju A=1.77 m2 (rura D=1.5m). Narysuj wykres slupkowy
> sumy strat przy Q=50 m3/s."*

**Uzyte funkcje:** `src.losses.minor_loss()`

In [ ]:
D_pipe = 1.5
A_pipe = np.pi * D_pipe**2 / 4

elements = [
    ('Wlot zaokraglony', 0.15),
    ('Kolano 90° (r/D=2)', 0.25),
    ('Kolano 90° (r/D=2)', 0.25),
    ('Zasuwa motylkowa', 0.3),
]

Q_test = 50.0
print(f'Straty miejscowe przy Q={Q_test} m3/s, D={D_pipe}m (A={A_pipe:.2f} m2):')
names, values = [], []
for name, xi in elements:
    dH = minor_loss(Q_test, A_pipe, xi)
    print(f'  {name:30s} (xi={xi:.2f}): {dH:.4f} m = {dH*100:.2f} cm')
    names.append(f'{name} (xi={xi})')
    values.append(float(dH))

print(f'  {"SUMA":30s}         : {sum(values):.4f} m = {sum(values)*100:.2f} cm')

fig = go.Figure(go.Bar(x=names, y=[v*100 for v in values], marker_color='royalblue'))
fig.update_layout(
    title=f'Straty miejscowe przy Q={Q_test} m3/s',
    yaxis_title='DeltaH [cm]', height=400,
)
fig.show()

---
## Krok 4: Spirala turbiny (komora spiralna)

Komora spiralna doprowadza wode do wirnika turbiny rownomaiernie
po obwodzie. Straty zaleza od jakosci wykonania:

$$\Delta H_{spirala} = \xi_{spirala} \cdot \frac{v_{wlot}^2}{2g}$$

Typowe $\xi$:
- Spirala dobrze zaprojektowana: 0.05--0.10
- Prosta komora: 0.10--0.20

**Prompt do LLM:**
> *"Napisz kod ktory pokaze straty w spirali dla xi=0.05, 0.10, 0.20."*

**Uzyte funkcje:** `src.losses.spiral_casing_loss()`

In [ ]:
A_spiral = 2.0  # m2 (wlot spirali)

fig = go.Figure()
for xi, color in [(0.05, 'green'), (0.10, 'royalblue'), (0.20, 'firebrick')]:
    dH = spiral_casing_loss(Q_demo, A_inlet=A_spiral, xi=xi)
    fig.add_trace(go.Scatter(x=Q_demo, y=dH*100, mode='lines',
        name=f'xi={xi}', line=dict(color=color, width=2)))

fig.update_layout(
    title=f'Straty w spirali turbiny (A_wlot={A_spiral} m2)',
    xaxis_title='Q [m3/s]', yaxis_title='DeltaH [cm]',
    height=400, hovermode='x unified',
)
fig.show()

---
## Krok 5: Rura ssawna (draft tube)

Rura ssawna to dyfuzor ponizej wirnika. Spowalnia wode i odzyskuje czesc
energii kinetycznej. Straty odnosimy do predkosci na wlocie (wyjscie z wirnika):

$$\Delta H_{rura} = \xi_{rura} \cdot \frac{v_{wlot}^2}{2g}$$

Typowe $\xi$:
- Rura stozikowa prosta: 0.15--0.25
- Rura kolanowa: 0.25--0.40
- Brak rury ssawnej (wylot z turbiny): $\xi \approx 1.0$

**Uwaga:** rura ssawna **zmniejsza** straty ogolne. Bez niej cala energia kinetyczna
za wirnikiem jest tracona ($\xi=1.0$). Dobra rura ssawna redukuje to do $\xi=0.2$.

**Prompt do LLM:**
> *"Porownaj straty: z rura ssawna (xi=0.25) vs bez rury (xi=1.0),
> przy A_inlet=0.8m2. Pokaz ile energii odzyskuje rura."*

**Uzyte funkcje:** `src.losses.draft_tube_loss()`

In [ ]:
A_runner_exit = 0.8  # m2 (wyjscie z wirnika)
A_dt_outlet = 3.0    # m2 (wylot rury ssawnej)

dH_with = draft_tube_loss(Q_demo, A_inlet=A_runner_exit, A_outlet=A_dt_outlet, xi=0.25)
dH_without = draft_tube_loss(Q_demo, A_inlet=A_runner_exit, A_outlet=A_runner_exit, xi=1.0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=Q_demo, y=dH_without, mode='lines',
    name='Bez rury ssawnej (xi=1.0)', line=dict(color='firebrick', width=2)))
fig.add_trace(go.Scatter(x=Q_demo, y=dH_with, mode='lines',
    name='Z rura ssawna (xi=0.25)', line=dict(color='green', width=2)))
fig.add_trace(go.Scatter(x=Q_demo, y=dH_without - dH_with, mode='lines',
    name='Odzyskana energia', line=dict(color='royalblue', width=2, dash='dot')))

fig.update_layout(
    title='Rura ssawna -- porownanie strat',
    xaxis_title='Q [m3/s]', yaxis_title='DeltaH [m]',
    height=400, hovermode='x unified',
)
fig.show()

Q_test = 50.0
print(f'Przy Q={Q_test} m3/s:')
print(f'  Bez rury ssawnej: {float(draft_tube_loss(Q_test, A_runner_exit, A_dt_outlet, 1.0)):.3f} m')
print(f'  Z rura ssawna:    {float(draft_tube_loss(Q_test, A_runner_exit, A_dt_outlet, 0.25)):.3f} m')

---
## Krok 6: Skladanie modeli -- wybor komponentow

Dla konkretnej instalacji student **recznie wybiera** ktore komponenty strat
dotycza jego przypadku. Kazdy komponent to funkcja `Q -> DeltaH`.

**Przyklady konfiguracji:**

| Typ instalacji | Komponenty |
|----------------|------------|
| Rurociag + turbina Kaplana | krata, tarcie rury, kolana, zasuwa, spirala, rura ssawna |
| Kanal otwarty + turbina | krata, spirala, rura ssawna |
| Derivacyjna z dlugim rurociagiem | krata, tarcie (dlugi), wiele kolan, zasuwa, spirala, rura ssawna |

**Prompt do LLM:**
> *"Napisz kod ktory zlozy model strat dla przykladowej instalacji rurociagowej:
> krata + rurociag (D=1.5m, L=50m) + 2 kolana + zasuwa + spirala + rura ssawna.
> Oblicz spad netto H_net(Q) i narysuj wykres H_gross vs H_net."*

**Uzyte funkcje:** `src.losses.total_head_loss()`, `src.losses.net_head()`

In [ ]:
# === KONFIGURACJA STRAT DLA INSTALACJI RUROCIAGOWEJ ===
# Student modyfikuje ta liste w zaleznosci od swojego przypadku

D_pipe = 1.5
A_pipe = np.pi * D_pipe**2 / 4

loss_fns = [
    lambda Q: trash_rack_loss(Q, A_rack=4.0, bar_width=0.012, bar_spacing=0.05),
    lambda Q: pipe_friction_loss(Q, D=D_pipe, L=50.0, k_s=0.001),
    lambda Q: minor_loss(Q, A_pipe, xi=0.15),   # wlot zaokraglony
    lambda Q: minor_loss(Q, A_pipe, xi=0.25),   # kolano 1
    lambda Q: minor_loss(Q, A_pipe, xi=0.25),   # kolano 2
    lambda Q: minor_loss(Q, A_pipe, xi=0.3),    # zasuwa motylkowa
    lambda Q: spiral_casing_loss(Q, A_inlet=2.0, xi=0.10),
    lambda Q: draft_tube_loss(Q, A_inlet=0.8, A_outlet=3.0, xi=0.25),
]

# Oblicz spad netto
H_gross = 6.0  # m
Q_range = np.linspace(5, 150, 200)
dH_total = total_head_loss(Q_range, loss_fns)
H_net_arr = net_head(H_gross, Q_range, loss_fns)

# Wykres
fig = go.Figure()
fig.add_trace(go.Scatter(x=Q_range, y=np.full_like(Q_range, H_gross),
    mode='lines', name='H brutto', line=dict(color='gray', width=1, dash='dash')))
fig.add_trace(go.Scatter(x=Q_range, y=H_net_arr,
    mode='lines', name='H netto', line=dict(color='royalblue', width=2.5)))
fig.add_trace(go.Scatter(x=Q_range, y=dH_total,
    mode='lines', name='Suma strat', line=dict(color='firebrick', width=2)))

fig.update_layout(
    title='Spad brutto vs netto (instalacja rurociagowa)',
    xaxis_title='Q [m3/s]', yaxis_title='H [m]',
    height=450, hovermode='x unified',
)
fig.show()

# Podsumowanie przy typowym przeplywie
Q_test = 50.0
dH = float(total_head_loss(Q_test, loss_fns))
H_n = float(net_head(H_gross, Q_test, loss_fns))
print(f'Przy Q = {Q_test} m3/s:')
print(f'  H brutto = {H_gross:.2f} m')
print(f'  Suma strat = {dH:.3f} m ({dH/H_gross*100:.1f}%)')
print(f'  H netto = {H_n:.3f} m')

**Prompt do LLM:**
> *"Narysuj wykres slupkowy rozkladu strat po komponentach przy Q=50 m3/s.
> Pokaz udzial procentowy kazdego elementu."*

In [ ]:
# Rozklad strat po komponentach
Q_test = 50.0
component_names = [
    'Krata wlotowa', 'Tarcie rurociagu', 'Wlot',
    'Kolano 1', 'Kolano 2', 'Zasuwa',
    'Spirala', 'Rura ssawna',
]
component_losses = [float(fn(Q_test)) for fn in loss_fns]
total = sum(component_losses)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'bar'}, {'type': 'pie'}]],
    subplot_titles=['Straty [cm]', 'Udzial procentowy'])

fig.add_trace(go.Bar(
    x=component_names, y=[v*100 for v in component_losses],
    marker_color='royalblue',
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=component_names, values=component_losses,
    textinfo='label+percent', hole=0.3,
), row=1, col=2)

fig.update_layout(
    title=f'Rozklad strat hydraulicznych przy Q={Q_test} m3/s (suma={total*100:.1f} cm)',
    height=450, showlegend=False,
)
fig.show()

---
## Podsumowanie

W tym notebooku przedstawiono modele strat hydraulicznych:

| Model | Wzor | Modul |
|-------|------|-------|
| Krata wlotowa | Kirschmer | `losses.trash_rack_loss()` |
| Tarcie rurociagu | Darcy-Weisbach + Swamee-Jain | `losses.pipe_friction_loss()` |
| Straty miejscowe | xi * v2/(2g) | `losses.minor_loss()` |
| Spirala turbiny | xi * v2/(2g) | `losses.spiral_casing_loss()` |
| Rura ssawna | xi * v2/(2g) | `losses.draft_tube_loss()` |

Modele sklada sie w liste `loss_fns` odpowiednia dla danego typu instalacji.
Funkcja `net_head(H_gross, Q, loss_fns)` oblicza spad netto.

**Lista `loss_fns` zostanie przekazana do notebooka 10 (obliczenia produkcji).**

**Dalej:** notebook 07 -- model sprawnosci turbiny